In [ ]:
# --- Cell 1/5: setup --------------------------------------------------------
REPO_URL = "<SET_REPO_URL>"   # <-- paste your repo URL here (no remote exists yet;
                              #     set this once you have pushed the branch)
!git clone {REPO_URL} loralink && cd loralink && pip -q install -r loralink_reviewer_response/requirements-colab.txt
%cd loralink
import sys; sys.path.insert(0, ".")
!python loralink_reviewer_response/patch/checksums.py --update && python loralink_reviewer_response/patch/checksums.py --verify

In [ ]:
# --- Cell 2/5: model download ---------------------------------------------
from huggingface_hub import snapshot_download
MODEL = "microsoft/phi-1_5"
snapshot_download(MODEL, local_dir=f"./models/{MODEL}",
                  allow_patterns=["*.json", "*.txt", "*.model", "*.safetensors", "*.bin",
                                  "merges.txt", "vocab.json", "tokenizer*"])

In [ ]:
# --- Cell 3/5: params (edit ACCOUNT_TAG and SHARD only) -----------------
ACCOUNT_TAG = "acct1"              # unique per Gmail account
SHARD       = "e2e:0"  # the only knob besides ACCOUNT_TAG
WALL_BUDGET_MIN = 32
import time; _NB_START = time.time()
def budget_left(): return WALL_BUDGET_MIN * 60 - (time.time() - _NB_START)

In [ ]:
# --- Cell 4/5: body -- compression ON/OFF vs centralized reference ----------
from loralink_reviewer_response.cluster_launch import run_cluster
from loralink_reviewer_response.eval_quality import evaluate_adapter

PER_RUN_ESTIMATE = 420
ds, seed = SHARD.split(":"); seed = int(seed)
ARMS = ["ON", "OFF", "reference"]
PLANNED, DONE = len(ARMS), 0
sys_csv = f"results_qsys_{ACCOUNT_TAG}.csv"
q_csv = f"results_quality_{ACCOUNT_TAG}.csv"

for arm in ARMS:
    if budget_left() < PER_RUN_ESTIMATE:
        print("budget exhausted, stopping"); break
    nw = 1 if arm == "reference" else 2
    adir = f"adapters/{ds}_s{seed}_{arm}"
    run_cluster(nw, ds, seed, model=MODEL, num_samples=50, epochs=1,
                compression=(arm != "OFF"), eval_holdout=200, strategy="smart",
                tag=f"q-{ds}-s{seed}-{arm}", results_csv=sys_csv,
                save_adapters_to=adir)
    evaluate_adapter(MODEL, adir, ds, arm=arm, seed=seed, limit=200, out_csv=q_csv)
    DONE += 1
print(f"done {DONE}/{PLANNED}")


In [ ]:
# --- Cell 5/5: download ---------------------------------------------------
import json, glob
from google.colab import files
json.dump({"tag": ACCOUNT_TAG, "shard": SHARD, "done": DONE, "planned": PLANNED,
           "checksums": open("loralink_reviewer_response/patch/SHA256SUMS").read()},
          open(f"run_manifest_{ACCOUNT_TAG}.json", "w"), indent=2)
# main.py writes per-batch rows to results_<kind>_<tag>.csv and the single summary
# row to results_<kind>_<tag>.summary.csv (schemas differ) -- grab both.
for f in (glob.glob(f"results_*_{ACCOUNT_TAG}.csv")
          + glob.glob(f"results_*_{ACCOUNT_TAG}.summary.csv")
          + [f"run_manifest_{ACCOUNT_TAG}.json"]):
    files.download(f)